In [1]:
import numpy as np
import matplotlib.pyplot as plt
import dedalus.public as d3
from ipywidgets import interact
from scipy.ndimage import gaussian_filter


In [2]:
#------------------ VARIABILI
Lx = 0.010
Lz = 0.00095
Nx = 192
Nz = 96
dealias = 3/2

g = 9.81

c0 = 0.40
phi0 = 0.01

kB = 1.380649e-23
T = 298.15
a_colloid = 11e-9

rho_w = 997.0
rho_L = 2200.0
beta_rho = 230.0

Ds0 = 1.025e-9
eps_Ds = -1.308e-9

eta_w = 0.00089
eta_gly = 0.945

w_eta = 0.705
x_eta = 2.0

alpha_dp = 125.0         #coeff diffusioforetico

rho_ref = rho_w + 0.5*(rho_L-rho_w)*phi0 + 0.5*beta_rho*c0

Ds_ref = Ds0
Dc_ref = kB*T/(6*np.pi*eta_w*a_colloid)
nu_ref = eta_w/rho_ref
rho_s_ref = rho_w

dt_sim = 1e-4
t_end = 1500.0
save_every = 5000

noise_amp_c = 1e-6
noise_amp_phi = 1e-7
interface_width = 0.05*Lz

print("rho_ref =", rho_ref)
print("Dc_ref =", Dc_ref)
print("Ds_ref/Dc_ref =", Ds_ref/Dc_ref)

rho_ref = 1049.0149999999999
Dc_ref = 2.2306646322222893e-11
Ds_ref/Dc_ref = 45.950430431976166


In [3]:
%%time
coords = d3.CartesianCoordinates("x", "z")
dist = d3.Distributor(coords, dtype=np.float64)

xbasis = d3.RealFourier(coords["x"], size=Nx, bounds=(0, Lx), dealias=dealias)  #con realfourier stai dicendo che hai un canale infinito ripetuto orizzontalmente
zbasis = d3.ChebyshevT(coords["z"], size=Nz, bounds=(0, Lz), dealias=dealias)

x, z = dist.local_grids(xbasis, zbasis)
ex, ez = coords.unit_vector_fields(dist)
p = dist.Field(name="p", bases=(xbasis, zbasis))
c = dist.Field(name="c", bases=(xbasis, zbasis))
phi = dist.Field(name="phi", bases=(xbasis, zbasis))
u = dist.VectorField(coords, name="u", bases=(xbasis, zbasis))

tau_p = dist.Field(name="tau_p")
tau_c1 = dist.Field(name="tau_c1", bases=xbasis)
tau_c2 = dist.Field(name="tau_c2", bases=xbasis)
tau_phi1 = dist.Field(name="tau_phi1", bases=xbasis)
tau_phi2 = dist.Field(name="tau_phi2", bases=xbasis)
tau_u1 = dist.VectorField(coords, name="tau_u1", bases=xbasis)
tau_u2 = dist.VectorField(coords, name="tau_u2", bases=xbasis)

lift_basis = zbasis.derivative_basis(1)
lift = lambda A: d3.Lift(A, lift_basis, -1)

grad_u = d3.grad(u) + ez*lift(tau_u1)
grad_c = d3.grad(c) + ez*lift(tau_c1)
grad_phi = d3.grad(phi) + ez*lift(tau_phi1)

rho_s_f = dist.Field(name="rho_s_f", bases=(xbasis, zbasis))
rho_f   = dist.Field(name="rho_f", bases=(xbasis, zbasis))
Ds_f    = dist.Field(name="Ds_f", bases=(xbasis, zbasis))
Dc_f    = dist.Field(name="Dc_f", bases=(xbasis, zbasis))
nu_f    = dist.Field(name="nu_f", bases=(xbasis, zbasis))

def update_coefficients():
    cg = np.clip(c["g"], 1e-6, 0.999)
    phig = np.clip(phi["g"], 0.0, 0.6)

    rho_s_g = rho_w + beta_rho*cg
    rho_g = rho_w + (rho_L-rho_w)*phig + beta_rho*(1-phig)*cg  #la densità aumenta sia per presenza di colloidi che di glicerolo

    Ds_g = Ds0 + eps_Ds*cg
    Ds_g = np.maximum(Ds_g, 1e-12)

    gamma_g = 1 - cg + (w_eta*x_eta*cg*(1-cg))/(w_eta*cg + x_eta*(1-cg))
    eta_local = np.exp(gamma_g*np.log(eta_w) + (1-gamma_g)*np.log(eta_gly))

    Dc_g = kB*T/(6*np.pi*eta_local*a_colloid)
    nu_g = eta_w/rho_ref   # per ora viscosità costante nella quantità di moto

    rho_s_f["g"] = rho_s_g
    rho_f["g"] = rho_g
    Ds_f["g"] = Ds_g
    Dc_f["g"] = Dc_g
    nu_f["g"] = nu_g



CPU times: user 7.37 ms, sys: 1.91 ms, total: 9.29 ms
Wall time: 11.3 ms


NameError: name 'Nx' is not defined

In [4]:
%%time
problem = d3.IVP(
    [p, c, phi, u, tau_p, tau_c1, tau_c2,
     tau_phi1, tau_phi2, tau_u1, tau_u2],
    namespace=locals()
)

problem.add_equation("trace(grad_u) + tau_p = 0")

problem.add_equation(                                       #equazione di glicerolo per trasporto convettico e diffusione con coeff diffusivo variabile
    "dt(c) - Ds_ref*div(grad_c) + lift(tau_c2) = "
    "- u@grad(c) "
    "+ (1/rho_s_ref)*div((Ds_f*rho_s_f - Ds_ref*rho_s_ref)*grad_c)"
)

problem.add_equation(                                       #migrazione guidata dal gradiente di glicerolo (diffusioforesi e osmotic drift delle nanoparticelle)
    "dt(phi) - Dc_ref*div(grad_phi) + lift(tau_phi2) = "
    "- u@grad(phi) "
    "+ div((Dc_f - Dc_ref)*grad_phi) "
    "+ div((alpha_dp/rho_w)*Dc_f*phi*grad_c)"
)
    
problem.add_equation(                                       #equazione della quantità di moto (Navier-Stokes incomprimibile in approssimazione di Boussinesq generalizzata)
    "dt(u) - nu_ref*div(grad_u) + grad(p)/rho_ref + lift(tau_u2) = "
    "- u@grad(u) "
    "- g*(rho_f-rho_ref)/rho_ref*ez" #forza di galleggiamento
) #se la densità locale aumento il fluido tende a scendere, else sale

##CONDIZIONI AL CONTORNO
problem.add_equation("ez@grad_c(z=0) = 0")        #stai dicendo che grad_c e grad_phi = 0 quindi non hai nessun flusso di massa attraverso le pareti
problem.add_equation("ez@grad_c(z=Lz) = 0")

problem.add_equation("ez@grad_phi(z=0) = 0")
problem.add_equation("ez@grad_phi(z=Lz) = 0")

problem.add_equation("u(z=0) = 0")               #no-slip walls
problem.add_equation("u(z=Lz) = 0")

problem.add_equation("integ(p) = 0")

CPU times: user 55.7 ms, sys: 3.59 ms, total: 59.3 ms
Wall time: 58.3 ms


{'eqn': Integrate(Integrate(<Field 5212457872>)),
 'LHS': Integrate(Integrate(<Field 5212457872>)),
 'RHS': 0,
 'condition': 'True',
 'tensorsig': (),
 'dtype': numpy.float64,
 'valid_modes': array([[ True]]),
 'M': 0,
 'L': Integrate(Integrate(<Field 5212457872>)),
 'F': <Field 4702055824>,
 'domain': <dedalus.core.domain.Domain at 0x136afc310>,
 'matrix_dependence': array([ True,  True]),
 'matrix_coupling': array([False,  True])}

In [5]:
solver = problem.build_solver(d3.RK222)
solver.stop_sim_time = t_end
print("Solver costruito")

2026-06-03 21:59:45,201 subsystems 0/1 INFO :: Building subproblem matrices 1/96 (~1%) Elapsed: 0s, Remaining: 11s, Rate: 8.9e+00/s
2026-06-03 21:59:45,593 subsystems 0/1 INFO :: Building subproblem matrices 10/96 (~10%) Elapsed: 1s, Remaining: 4s, Rate: 2.0e+01/s
2026-06-03 21:59:46,039 subsystems 0/1 INFO :: Building subproblem matrices 20/96 (~21%) Elapsed: 1s, Remaining: 4s, Rate: 2.1e+01/s
2026-06-03 21:59:46,488 subsystems 0/1 INFO :: Building subproblem matrices 30/96 (~31%) Elapsed: 1s, Remaining: 3s, Rate: 2.1e+01/s
2026-06-03 21:59:46,938 subsystems 0/1 INFO :: Building subproblem matrices 40/96 (~42%) Elapsed: 2s, Remaining: 3s, Rate: 2.2e+01/s
2026-06-03 21:59:47,386 subsystems 0/1 INFO :: Building subproblem matrices 50/96 (~52%) Elapsed: 2s, Remaining: 2s, Rate: 2.2e+01/s
2026-06-03 21:59:47,826 subsystems 0/1 INFO :: Building subproblem matrices 60/96 (~62%) Elapsed: 3s, Remaining: 2s, Rate: 2.2e+01/s
2026-06-03 21:59:48,274 subsystems 0/1 INFO :: Building subproblem mat

In [6]:
%%time
H = 0.5*(1 - np.tanh((z - Lz/2)/interface_width))   #qui spiega che sotto miscela e sopra acqua

c["g"] = c0*H
phi["g"] = phi0*H

noise_c = dist.Field(name="noise_c", bases=(xbasis, zbasis))  #aggiunta di rumore per rompere la simmetria
noise_phi = dist.Field(name="noise_phi", bases=(xbasis, zbasis))

noise_c.fill_random("g", seed=42, distribution="normal", scale=1.0)
noise_phi.fill_random("g", seed=43, distribution="normal", scale=1.0)

window = (z/Lz)*(1 - z/Lz)

c["g"] += noise_amp_c*gaussian_filter(noise_c["g"], sigma=1)*window
phi["g"] += noise_amp_phi*gaussian_filter(noise_phi["g"], sigma=1)*window

c["g"] = np.clip(c["g"], 0, 1)
phi["g"] = np.clip(phi["g"], 0, 0.6)

u["g"][0] = 0.0
u["g"][1] = 0.0

print("c min/max =", np.min(c["g"]), np.max(c["g"]))
print("phi min/max =", np.min(phi["g"]), np.max(phi["g"]))

c min/max = 0.0 0.40000004116962395
phi min/max = 0.0 0.010000006933098621
CPU times: user 325 ms, sys: 4.09 ms, total: 329 ms
Wall time: 332 ms


In [ ]:
%%time
times = []
c_all = []
phi_all = []
rho_all = []
Ux_all = []
Uz_all = []
umax_all = []
wmax_all = []
ke_all = []
cfl_all = []

dx = Lx/Nx
dz_min = Lz/(Nz**2)
update_coefficients()
while solver.proceed:
    update_coefficients()
    solver.step(dt_sim)

    Ux_now = u["g"][0]
    Uz_now = u["g"][1]

    umax = np.max(np.abs(Ux_now))
    wmax = np.max(np.abs(Uz_now))
    ke = np.mean(Ux_now**2 + Uz_now**2)

    cfl = dt_sim*(umax/dx + wmax/max(dz_min, 1e-12))

    if solver.iteration % save_every == 0:
        #rho_now = rho_w + (rho_L-rho_w)*phi["g"] + beta*(1-phi["g"])*c["g"]
        cg = np.clip(c["g"], 1e-6, 0.999)
        phig = np.clip(phi["g"], 0.0, 0.6)
        rho_now = rho_w + (rho_L-rho_w)*phig + beta_rho*(1-phig)*cg

        
        times.append(solver.sim_time)
        c_all.append(c["g"].copy())
        phi_all.append(phi["g"].copy())
        rho_all.append(rho_now.copy())
        Ux_all.append(Ux_now.copy())
        Uz_all.append(Uz_now.copy())
        umax_all.append(umax)
        wmax_all.append(wmax)
        ke_all.append(ke)
        cfl_all.append(cfl)

        print(
            f"iter={solver.iteration:6d}, "
            f"t={solver.sim_time:9.2f} s, "
            f"umax={umax:.3e}, "
            f"wmax={wmax:.3e}, "
            f"KE={ke:.3e}, "
            f"CFL~{cfl:.2e}"
        )

print("Simulazione finita")

iter=  5000, t=     0.50 s, umax=4.452e-10, wmax=4.844e-10, KE=1.405e-20, CFL~4.71e-07


In [ ]:
%%time
times = np.array(times)
c_all = np.array(c_all)  
phi_all = np.array(phi_all)
rho_all = np.array(rho_all)     #densità
Ux_all = np.array(Ux_all)
Uz_all = np.array(Uz_all)
umax_all = np.array(umax_all)   #massimo della velocità
wmax_all = np.array(wmax_all)   #massimo della velocità
ke_all = np.array(ke_all)       #en cinetica
cfl_all = np.array(cfl_all)     #per controllare la stabilità numerica

x_plot, z_plot = dist.local_grids(xbasis, zbasis, scales=dealias)
X, Z = np.meshgrid(x_plot[:, 0], z_plot[0, :], indexing="ij")

print("shape =", c_all.shape)

In [ ]:
%%time
def plot_frame(n=0):
    speed = np.sqrt(Ux_all[n]**2 + Uz_all[n]**2)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

    im0 = ax[0].contourf(X, Z, c_all[n], levels=40)
    plt.colorbar(im0, ax=ax[0], label="c glicerolo")
    ax[0].set_title(f"c, t={times[n]:.1f} s")

    im1 = ax[1].contourf(X, Z, phi_all[n], levels=40)
    plt.colorbar(im1, ax=ax[1], label="phi colloide")
    ax[1].set_title("colloide")

    im2 = ax[2].contourf(X, Z, speed, levels=30)
    skipx, skipz = 8, 5
    ax[2].quiver(
        X[::skipx, ::skipz],
        Z[::skipx, ::skipz],
        Ux_all[n][::skipx, ::skipz],
        Uz_all[n][::skipx, ::skipz],
        pivot="mid"
    )
    plt.colorbar(im2, ax=ax[2], label="|u| [m/s]")
    ax[2].set_title("velocità")

    for a in ax:
        a.set_xlabel("x [m]")
        a.set_ylabel("z [m]")

    plt.show()

interact(plot_frame, n=(0, max(len(times)-1, 0), 1))

plt.figure(figsize=(7, 4))
plt.plot(times, umax_all, label="max |u_x|")
plt.plot(times, wmax_all, label="max |u_z|")
plt.xlabel("t [s]")
plt.ylabel("velocità massima [m/s]")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.semilogy(times, ke_all + 1e-300)
plt.xlabel("t [s]")
plt.ylabel("<|u|²> [m²/s²]")
plt.grid(True)
plt.tight_layout()
plt.show()

def density_inversion_metric(rho_frame):
    prof = rho_frame.mean(axis=0)
    best = 0.0

    for i in range(len(prof)):
        dr_above = np.max(prof[i:]) - prof[i]
        if dr_above > best:
            best = dr_above

    return best

inv = np.array([density_inversion_metric(r) for r in rho_all])

plt.figure(figsize=(7, 4))
plt.plot(times, inv)
plt.xlabel("t [s]")
plt.ylabel("max inversione densità media [kg/m³]")
plt.grid(True)
plt.tight_layout()
plt.show()

n = -1

plt.figure(figsize=(7, 4))
plt.plot(c_all[n].mean(axis=0)/max(c0, 1e-30), z_plot[0, :]/Lz, label="c/c0")
plt.plot(phi_all[n].mean(axis=0)/max(phi0, 1e-30), z_plot[0, :]/Lz, label="phi/phi0")
plt.xlabel("profilo medio normalizzato")
plt.ylabel("z/Lz")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()